In [1]:
!pip install scikit-learn pandas joblib

In [23]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier, AdaBoostClassifier

from sklearn.metrics import confusion_matrix, classification_report

In [2]:
from google.colab import files
uploaded=files.upload()

Saving SMSSpamCollection to SMSSpamCollection


In [7]:
import os
os.listdir()

['.config', 'SMSSpamCollection', 'sample_data']

In [16]:
df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label","message"]
    )
df.dropna(inplace=True)
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [18]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["label"]=le.fit_transform(df["label"])
df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [19]:
X = df["message"]
y = df["label"]

In [15]:
df.columns

Index(['label', 'messsage'], dtype='object')

In [25]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1,2)
)

In [26]:
nb = MultinomialNB()
lr = LogisticRegression(max_iter=1000)
svm=LinearSVC()

In [27]:
soft_voting = VotingClassifier(
    estimators=[
        ("nb",nb),
        ("lr",lr)
    ],
    voting = "soft"
)

In [29]:
hard_voting = VotingClassifier(
    estimators=[
        ("nb",nb) , ("lr",lr)
     ],
    voting="hard"
)

In [30]:
stacking = StackingClassifier(
    estimators=[("nb", nb), ("lr", lr)],
    final_estimator=LogisticRegression()
)

In [31]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=100,
    learning_rate=0.5
)

In [35]:
models = {
    "NaiveBayes": (nb, True),
    "LogisticRegression": (lr, True),
    "LinearSVM": (svm, False),
    "HardVoting": (hard_voting, False),
    "SoftVoting": (soft_voting, True),
    "Stacking": (stacking, True),
    "AdaBoost_Stumps": (ada, True)
}

In [34]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [36]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

results = []

for name, (model, has_auc) in models.items():

    pipeline = Pipeline([
        ("tfidf", tfidf),
        ("clf", model)
    ])

    scoring = ["precision", "recall", "f1"]
    if has_auc:
        scoring.append("roc_auc")

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring
    )

    row = {
        "Model": name,
        "Precision_mean": scores["test_precision"].mean(),
        "Recall_mean": scores["test_recall"].mean(),
        "F1_mean": scores["test_f1"].mean()
    }

    if has_auc:
        row["ROC_AUC_mean"] = scores["test_roc_auc"].mean()
    else:
        row["ROC_AUC_mean"] = "Not Applicable"

    results.append(row)

In [37]:
comparison_df = pd.DataFrame(results)
comparison_df.to_csv("ensemble_comparison.csv", index=False)
comparison_df

,Model,Precision_mean,Recall_mean,F1_mean,ROC_AUC_mean
0,NaiveBayes,0.998333,0.808555,0.893303,0.987349
1,LogisticRegression,0.986205,0.769700,0.864515,0.990505
2,LinearSVM,0.982719,0.906273,0.942860,Not Applicable
3,HardVoting,1.000000,0.726864,0.841580,Not Applicable
4,SoftVoting,0.993273,0.811239,0.892914,0.990922
5,Stacking,0.983726,0.892877,0.936002,0.990465
6,AdaBoost_Stumps,0.988854,0.236904,0.381577,0.8971


In [38]:
final_model = Pipeline([
    ("tfidf", tfidf),
    ("clf", soft_voting)
])

In [39]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

confusion_matrix(y_test, y_pred)

array([[966,   0],
       [ 30, 119]])

In [41]:
final_model.fit(X, y)

preds = final_model.predict(X)
probs = final_model.predict_proba(X)[:, 1]

final_df = pd.DataFrame({
    "message": df["message"],
    "Actual": y,
    "Predicted": preds,
    "Probability": probs
})

final_df.to_csv("final_model_predictions.csv", index=False)